In [ ]:
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score,
    confusion_matrix, classification_report
)

warnings.filterwarnings('ignore')

# Semilla global
RANDOM_SEED = 42

# Nombres de columnas — NSL-KDD (41 features + label + difficulty)
# Fuente: Aljawarneh 2018, Tabla 2 + descripción del dataset
COL_NAMES = [
    'duration', 'protocol_type', 'service', 'flag',
    'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
    'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
    'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
    'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate',
    'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate',
    'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate',
    'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate',
    'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
    'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'label', 'difficulty'
]

# Features seleccionadas por Information Gain > 0.40 (Table 4, Aljawarneh 2018) ─
# Features 5, 3, 6, 4, 30, 29, 33, 34 en numeración del paper
SELECTED_FEATURES = [
    'src_bytes',              # feature 5  
    'service',                # feature 3  
    'dst_bytes',              # feature 6
    'flag',                   # feature 4  
    'diff_srv_rate',          # feature 30
    'same_srv_rate',          # feature 29
    'dst_host_srv_count',     # feature 33
    'dst_host_same_srv_rate', # feature 34
]

# Clases — 5 categorías de NSL-KDD (Aljawarneh 2018, Section 5.2.A)
# 1=Normal, 2=DoS, 3=Probe, 4=R2L, 5=U2R
CLASS_NAMES = {1: 'Normal', 2: 'DoS', 3: 'Probe', 4: 'R2L', 5: 'U2R'}

# Localización del archivo
# Paper Section 5.2: "the detectors were trained using the NSL KDDTrain+20%
# made up of 25192 instances, 13449 normal and 11743 attack data"
def find_file(filename):
    search_paths = [
        (Path.home() / ".cache" / "kagglehub" / "datasets" / "hassan06" / "nslkdd" / "versions" / "1"),
        Path("."),
        Path.home() / "Downloads",
        Path.home() / "datasets",
        Path("/data"),
    ]
    for base in search_paths:
        candidate = base / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"No se encontró '{filename}'.\n"
        "Descárgalo de: https://www.kaggle.com/datasets/hassan06/nslkdd\n"
        "y colócalo en el directorio actual o en ~/Downloads/"
    )

df = pd.read_csv(
    find_file('KDDTrain+_20Percent.txt'),
    header=None,
    names=COL_NAMES
).drop(columns=['difficulty'])

print(f"Dataset   : NSL-KDD (KDDTrain+_20Percent.txt)")
print(f"Instancias: {len(df):,}")
print(f"Columnas  : {df.shape[1]}")
print(f"\nDistribución de ataques (label original):")
print(df['label'].value_counts().to_string())


Dataset   : NSL-KDD (KDDTrain+_20Percent.txt)
Instancias: 25,192
Columnas  : 42

Distribución de ataques (label original):
label
normal             13449
neptune             8282
ipsweep              710
satan                691
portsweep            587
smurf                529
nmap                 301
back                 196
teardrop             188
warezclient          181
pod                   38
guess_passwd          10
warezmaster            7
buffer_overflow        6
imap                   5
rootkit                4
multihop               2
phf                    2
ftp_write              1
land                   1
loadmodule             1
spy                    1


In [ ]:
# Paso 1: Encoding de variables nominales (Section 5.2.A)
# El paper especifica: "specific values were assigned to each variable"
# TCP=1, UDP=2, ICMP=3 — único mapeo explícito mencionado en el paper
protocol_map = {'tcp': 1, 'udp': 2, 'icmp': 3}
df['protocol_type'] = df['protocol_type'].str.lower().map(protocol_map).fillna(0)

# service y flag: LabelEncoder sobre el dataset completo
le_service = LabelEncoder()
le_flag    = LabelEncoder()
df['service'] = le_service.fit_transform(df['service'].astype(str))
df['flag']    = le_flag.fit_transform(df['flag'].astype(str))

# Paso 2: Mapeo de etiquetas a 5 clases (Section 5.2.A)
# Aljawarneh 2018 agrupa los 39 tipos de ataque en 4 clases + Normal
DOS   = {'back','land','neptune','pod','smurf','teardrop',
         'apache2','udpstorm','processtable','mailbomb'}
PROBE = {'ipsweep','nmap','portsweep','satan','mscan','saint'}
R2L   = {'ftp_write','guess_passwd','imap','multihop','phf','spy',
         'warezclient','warezmaster','sendmail','named',
         'snmpgetattack','snmpguess','xlock','xsnoop','worm'}
U2R   = {'buffer_overflow','loadmodule','perl','rootkit',
         'httptunnel','ps','sqlattack','xterm'}

def map_label(label):
    label = label.lower().strip()
    if label == 'normal': return 1
    if label in DOS:      return 2
    if label in PROBE:    return 3
    if label in R2L:      return 4
    if label in U2R:      return 5
    return 2  # ataque no catalogado → DoS por defecto

df['label'] = df['label'].apply(map_label)

# Paso 3: Selección de features (Table 4, Aljawarneh 2018)
# Solo las 8 features con Information Gain > 0.40
X = df[SELECTED_FEATURES].values
y = df['label'].values

# Paso 4: Split 80/20 estratificado (Section 5, paso 2)
# "Apportioning the dataset into 20% test and 80% train"
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=y
)

# Paso 5: Normalización Min-Max (Section 5.2.B)
# "min-max normalization was used to normalize the features,
#  mapping all values to the [0,1] range"
scaler  = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

CLASSES     = sorted(np.unique(y_train))  # [1, 2, 3, 4, 5]

# Resumen del preprocesamiento
print(f"  Features seleccionadas : {SELECTED_FEATURES}")
print(f"  Normalización          : MinMaxScaler [0, 1]")
print(f"  Split                  : 80% train / 20% test")
print(f"  Train                  : {len(X_train):,} instancias")
print(f"  Test                   : {len(X_test):,} instancias")
print(f"\nDistribución por clase:")
print(f"  {'Clase':<10} {'Train':>8}  {'Test':>6}")
print(f"  {'─'*10} {'─'*8}  {'─'*6}")
for c in CLASSES:
    n_tr = (y_train == c).sum()
    n_te = (y_test  == c).sum()
    print(f"  {CLASS_NAMES[c]:<10} {n_tr:>8,}  {n_te:>6,}")


  Features seleccionadas : ['src_bytes', 'service', 'dst_bytes', 'flag', 'diff_srv_rate', 'same_srv_rate', 'dst_host_srv_count', 'dst_host_same_srv_rate']
  Normalización          : MinMaxScaler [0, 1]
  Split                  : 80% train / 20% test
  Train                  : 20,153 instancias
  Test                   : 5,039 instancias

Distribución por clase:
  Clase         Train    Test
  ────────── ────────  ──────
  Normal       10,759   2,690
  DoS           7,387   1,847
  Probe         1,831     458
  R2L             167      42
  U2R               9       2


In [ ]:
CLASSIFIERS = {
    'KNN'     : KNeighborsClassifier(metric='euclidean'),
    'LDA'     : LinearDiscriminantAnalysis(),

    'DT'      : DecisionTreeClassifier(random_state=RANDOM_SEED),
    'RF'      : RandomForestClassifier(random_state=RANDOM_SEED),
    'LR'      : LogisticRegression( max_iter=1000,random_state=RANDOM_SEED),
    'SVM'     : SVC(kernel='rbf',gamma=0.1,random_state=RANDOM_SEED),
    'ANN'     : MLPClassifier(activation='logistic',solver='sgd',max_iter=500,random_state=RANDOM_SEED),
    'AdaBoost': AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=3, random_state=RANDOM_SEED),
                    n_estimators=100, random_state=RANDOM_SEED, algorithm='SAMME'),
}

# Entrenamiento
predictions = {}
for name, clf in CLASSIFIERS.items():
    print(f"Entrenando {name}...", end=' ', flush=True)
    clf.fit(X_train, y_train)
    predictions[name] = clf.predict(X_test)
    print("listo.")

print("\nEntrenamiento completado.")


Entrenando KNN... 

listo.
Entrenando LDA... 

listo.
Entrenando DT... 

listo.
Entrenando RF... 

listo.
Entrenando LR... 

listo.
Entrenando SVM... 

listo.
Entrenando ANN... 

listo.
Entrenando AdaBoost... 

listo.

Entrenamiento completado.


In [4]:
# tabla de resultados: replica la Tabla I de Khare & Totaro (2020) + desglose por clase

# Tabla principal: Precision, Recall, F1, Accuracy (weighted average)
# Formato idéntico al Bloque 4 del notebook Khare_Totaro_2020.ipynb (DS2OS)
# Las métricas se calculan en modo 'weighted' como hace el paper original

SEP = '─' * 72

print("TABLA I — Khare & Totaro (2020): Clasificadores sobre NSL-KDD")
print(SEP)
print(f"{'Modelo':<10} {'Prec':>6} {'Rec':>6} {'F1':>6} {'Acc':>6}")
print(f"{'─'*10} {'─'*6} {'─'*6} {'─'*6} {'─'*6}")

for name, y_pred in predictions.items():
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec  = recall_score   (y_test, y_pred, average='weighted', zero_division=0)
    f1   = f1_score       (y_test, y_pred, average='weighted', zero_division=0)
    acc  = accuracy_score (y_test, y_pred)
    print(f"{name:<10} {prec:.3f}  {rec:.3f}  {f1:.3f}  {acc:.3f}")

print(SEP)
print("Métricas calculadas en modo 'weighted average' (igual que Khare & Totaro 2020)")

# Desglose por clase (Tabla 5 / Tabla 6 de Aljawarneh 2018)
# Para NSL-KDD se añade el desglose por clase usando las métricas de Aljawarneh:
# TP Rate (Recall por clase) y Accuracy por clase

def per_class_metrics(y_true, y_pred, classes):
    """Calcula TP Rate y Accuracy por clase — estilo Aljawarneh 2018 Tabla 5."""
    cm = confusion_matrix(y_true, y_pred, labels=classes)
    results = {}
    for i, c in enumerate(classes):
        TP = cm[i, i]
        FN = cm[i, :].sum() - TP
        support = cm[i, :].sum()
        tp_rate = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        acc_cls = TP / support   if support > 0    else 0.0
        results[c] = {'tp_rate': tp_rate, 'acc': acc_cls * 100, 'support': support}
    return results

print(f"\n{'─'*72}")
print("DESGLOSE POR CLASE — TP Rate y Accuracy por clase")
print(f"{'─'*72}")
print(f"{'Modelo':<10} {'Clase':<8} {'TP Rate':>8} {'Acc (%)':>9} {'Support':>9}")
print(f"{'─'*10} {'─'*8} {'─'*8} {'─'*9} {'─'*9}")

for name, y_pred in predictions.items():
    metrics = per_class_metrics(y_test, y_pred, CLASSES)
    for j, c in enumerate(CLASSES):
        m      = metrics[c]
        label  = name if j == 0 else ''
        print(f"{label:<10} {CLASS_NAMES[c]:<8} {m['tp_rate']:>8.3f} "
              f"{m['acc']:>8.1f}% {m['support']:>8,}")
    print(f"{'─'*10} {'─'*8} {'─'*8} {'─'*9} {'─'*9}")

# Classification report completo (referencia interna)
print(f"\n{'─'*72}")
print("CLASSIFICATION REPORT COMPLETO — Mejor modelo (AdaBoost)")
print(f"{'─'*72}")
target_names = [CLASS_NAMES[c] for c in CLASSES]
print(classification_report(
    y_test, predictions['AdaBoost'],
    labels=CLASSES,
    target_names=target_names,
    zero_division=0
))


TABLA I — Khare & Totaro (2020): Clasificadores sobre NSL-KDD
────────────────────────────────────────────────────────────────────────
Modelo       Prec    Rec     F1    Acc
────────── ────── ────── ────── ──────
KNN        0.971  0.971  0.971  0.971
LDA        0.870  0.858  0.863  0.858
DT         0.993  0.993  0.993  0.993
RF         0.996  0.996  0.996  0.996
LR         0.874  0.879  0.871  0.879
SVM        0.893  0.893  0.882  0.893
ANN        0.874  0.882  0.871  0.882
AdaBoost   0.988  0.988  0.988  0.988
────────────────────────────────────────────────────────────────────────
Métricas calculadas en modo 'weighted average' (igual que Khare & Totaro 2020)

────────────────────────────────────────────────────────────────────────
DESGLOSE POR CLASE — TP Rate y Accuracy por clase
────────────────────────────────────────────────────────────────────────
Modelo     Clase     TP Rate   Acc (%)   Support
────────── ──────── ──────── ───────── ─────────
KNN        Normal      0.980     98.